# Загрузка CSV через Dask в MS SQL Server

Тестовое задание X5 Retail Group для аналитика данных в FMCG.

Объединяем CSV одинаковой структуры в один Dask DataFrame, явно задаём типы и
учитываем российский формат чисел. Затем загружаем данные частями в SQL Server
и выполняем полную замену целевой таблицы.

Все функции находятся в этом notebook. Для повторного запуска откройте его
из распакованного проекта и выберите **Run All**. Исходный файл ожидается
в папке `data` рядом с папкой `notebooks`.


## Библиотеки и параметры

Для повторного запуска установите библиотеки.

```bash
pip install "dask[dataframe]" pandas pyarrow sqlalchemy pyodbc jupyterlab
```

Для работы с SQL Server нужен Microsoft ODBC Driver 18. В параметрах ниже
выполнение записи управляется переменной `RUN_SQL`. Сохранённый результат
получен при `RUN_SQL = False`.


In [1]:
from __future__ import annotations

import csv
import glob
import itertools
import logging
import math
import os
import re
from collections.abc import Iterator, Mapping
from pathlib import Path
from uuid import uuid4

import dask
import dask.dataframe as dd
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import sqlalchemy as sa
from sqlalchemy.dialects import mssql
from sqlalchemy.engine import Connection, Engine, URL
from sqlalchemy.exc import SQLAlchemyError



from tempfile import TemporaryDirectory

from IPython.display import display

pd.set_option("display.max_columns", 12)
pd.set_option("display.precision", 4)


In [2]:
def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "data" / "sales_data.csv").is_file():
            return candidate
    raise FileNotFoundError(
        "Не найден data/sales_data.csv. Откройте notebook из папки проекта."
    )


PROJECT_ROOT = find_project_root()
INPUT_DIRECTORY = PROJECT_ROOT / "data"
RU_EXAMPLES = PROJECT_ROOT / "examples" / "input_ru"
RUN_SQL = False
SQL_SCHEMA = "dbo"
SQL_TABLE = "fmcg_sales"

display(pd.DataFrame({
    "Параметр": ["Исходные CSV", "Примеры с десятичной запятой", "Запись в SQL Server"],
    "Значение": ["data/", "examples/input_ru/", "включена" if RUN_SQL else "отключена"],
}))


,Параметр,Значение
0,Исходные CSV,data/
1,Примеры с десятичной запятой,examples/input_ru/
2,Запись в SQL Server,отключена


## Схема данных

Идентификаторы храним строками, чтобы сохранить ведущие нули. Категория получает
тип `category`, количества — nullable `int64`, даты — календарный тип Arrow.
Цены хранятся как `decimal128(19, 4)`, что сохраняет точность до четырёх знаков
после запятой. Пустая строка становится пропуском.

Разделитель полей и десятичный знак передаём явно. Для российских CSV используем
`;` и `,`, для исходного файла тестового — `,` и `.`.


In [3]:
LOG = logging.getLogger(__name__)
SOURCE_PATH = "_source_path"
SALES_SCHEMA: dict[str, str] = {
    "product_id": "id",
    "product_name": "text",
    "category": "category",
    "purchase_price": "decimal",
    "selling_price": "decimal",
    "quantity_sold": "integer",
    "quantity_purchased": "integer",
    "losses": "integer",
    "date": "date",
    "store_id": "id",
}
ARROW_TYPES = {
    "decimal": pa.decimal128(19, 4),
    "integer": pa.int64(),
    "date": pa.date32(),
}
SQL_TYPES = {
    "id": mssql.NVARCHAR(64),
    "text": mssql.NVARCHAR(256),
    "category": mssql.NVARCHAR(256),
    "decimal": mssql.DECIMAL(19, 4),
    "integer": mssql.BIGINT(),
    "date": mssql.DATE(),
}


In [4]:
display(pd.DataFrame([
    {
        "Поле": column,
        "Тип в схеме": kind,
        "Тип SQL Server": str(SQL_TYPES[kind]),
    }
    for column, kind in SALES_SCHEMA.items()
]))


,Поле,Тип в схеме,Тип SQL Server
0,product_id,id,NVARCHAR(64)
1,product_name,text,NVARCHAR(256)
2,category,category,NVARCHAR(256)
3,purchase_price,decimal,"DECIMAL(19, 4)"
4,selling_price,decimal,"DECIMAL(19, 4)"
5,quantity_sold,integer,BIGINT
6,quantity_purchased,integer,BIGINT
7,losses,integer,BIGINT
8,date,date,DATE
9,store_id,id,NVARCHAR(64)


## Ленивое чтение каталога

Функция проверяет заголовки всех CSV верхнего уровня каталога, создаёт единый
DataFrame через `dd.read_csv` и задаёт преобразование каждого раздела.
Числа с неверным разделителем, некорректные даты и несоответствие схеме вызывают
ошибку. Точный файл и поле указаны в сообщении.

По умолчанию блок CSV занимает около 32 МиБ. Для файлов с переносами строк
внутри кавычек передайте `blocksize=None`. Каждый такой файл будет одним разделом.


In [5]:
def _csv_files(directory: Path) -> list[Path]:
    if not directory.is_dir():
        raise ValueError(f"Каталог не существует или недоступен: {directory}")
    files = sorted(
        path for path in directory.iterdir()
        if path.is_file() and path.suffix.lower() == ".csv"
    )
    if not files:
        raise ValueError(f"В каталоге нет файлов CSV: {directory}")
    return files


def _check_headers(
    files: list[Path], columns: list[str], sep: str, encoding: str
) -> None:
    for path in files:
        try:
            with path.open(encoding=encoding, newline="") as source:
                header = next(csv.reader(source, delimiter=sep, strict=True), [])
        except (OSError, UnicodeError, csv.Error) as error:
            raise ValueError(f"Не удалось прочитать заголовок {path.name}: {error}") from error
        if header != columns:
            raise ValueError(f"Неверный состав или порядок колонок в {path.name}: {header}")


In [6]:
def _type_partition(
    partition: pd.DataFrame,
    schema: Mapping[str, str],
    decimal: str,
) -> pd.DataFrame:
    files = ", ".join(Path(str(path)).name for path in partition[SOURCE_PATH].unique())
    if not isinstance(partition.index, pd.RangeIndex):
        raise ValueError(f"{files}: проверьте число полей и кавычки в строках CSV.")
    result = partition.drop(columns=SOURCE_PATH)
    for column, kind in schema.items():
        if kind not in ARROW_TYPES:
            continue
        values = result[column].str.strip()
        if kind == "decimal":
            is_number = values.str.fullmatch(
                rf"[+-]?[0-9]+(?:{re.escape(decimal)}[0-9]+)?", na=True
            )
            if not is_number.all():
                raise ValueError(
                    f"{files}, {column}: неверное число или десятичный разделитель "
                    f"(ожидается {decimal!r})."
                )
            if decimal == ",":
                values = values.str.replace(",", ".", regex=False)
        arrow_type = ARROW_TYPES[kind]
        try:
            typed = pc.cast(pa.array(values.array), arrow_type, safe=True)
        except pa.ArrowInvalid as error:
            raise ValueError(f"{files}, {column}: {str(error)[:200]}") from error
        result[column] = pd.Series(
            typed, index=result.index, dtype=pd.ArrowDtype(arrow_type)
        )
    return result


In [7]:
def read_csv_directory(
    directory: str | Path,
    *,
    schema: Mapping[str, str] = SALES_SCHEMA,
    sep: str = ";",
    decimal: str = ",",
    encoding: str = "utf-8-sig",
    blocksize: str | int | None = "32 MiB",
) -> dd.DataFrame:
    """Вернуть ленивый Dask DataFrame для всех CSV верхнего уровня каталога."""
    schema = dict(schema)
    if not schema or SOURCE_PATH in schema or set(schema.values()) - SQL_TYPES.keys():
        raise ValueError(f"Схема должна содержать типы из {', '.join(SQL_TYPES)}.")
    if len(sep) != 1 or decimal not in (".", ","):
        raise ValueError("Требуются односимвольный разделитель полей и десятичный знак ',' или '.'.")
    files = _csv_files(Path(directory).resolve())
    _check_headers(files, list(schema), sep, encoding)
    raw = dd.read_csv(
        [glob.escape(path.as_posix()) for path in files],
        sep=sep,
        encoding=encoding,
        blocksize=blocksize,
        dtype={
            column: "category" if kind == "category" else "string[pyarrow]"
            for column, kind in schema.items()
        },
        include_path_column=SOURCE_PATH,
        keep_default_na=False,
        na_values=[""],
        enforce=True,
        on_bad_lines="error",
    )
    # Метаданные типов известны до вычисления, категории определяются по разделам.
    meta = raw._meta.drop(columns=SOURCE_PATH)
    for column, kind in schema.items():
        if kind in ARROW_TYPES:
            meta[column] = pd.Series(dtype=pd.ArrowDtype(ARROW_TYPES[kind]))
    return raw.map_partitions(_type_partition, schema, decimal, meta=meta)


Для проверки и последующей записи вычисляем ограниченное число разделов
за один проход. Параметр `workers` задаёт число одновременно читаемых разделов.
Память зависит от размера текущих разделов и пакетов ODBC.


In [8]:
def _partitions(frame: dd.DataFrame, workers: int) -> Iterator[pd.DataFrame]:
    if workers < 1:
        raise ValueError("Число рабочих потоков должно быть положительным.")
    for start in range(0, frame.npartitions, workers):
        pending = frame.partitions[start:start + workers].to_delayed()
        yield from dask.compute(*pending, scheduler="threads", num_workers=workers)


## Два CSV с российским форматом чисел

В примере объединяем два файла продаж. Цены содержат десятичную запятую,
коды товара и магазина — ведущие нули. У одной строки отсутствует закупочная цена.
Для демонстрации код сам создаёт два CSV во временном каталоге,
если входных примеров нет.


In [9]:
RU_ROWS = [
    [
        ["001", "Молоко 1 л", "Молочные продукты", "54,32", "79,99", "51", "60", "4", "2023-10-02", "005"],
        ["002", "Хлеб 450 г", "Хлебобулочные изделия", "32,50", "45,90", "80", "90", "6", "2023-10-02", "005"],
    ],
    [
        ["001", "Молоко 1 л", "Молочные продукты", "54,32", "79,99", "63", "72", "5", "2023-10-03", "013"],
        ["002", "Хлеб 450 г", "Хлебобулочные изделия", "", "45,90", "77", "84", "6", "2023-10-03", "013"],
    ],
]


def write_ru_examples(directory: Path) -> None:
    for number, rows in enumerate(RU_ROWS, start=1):
        with (directory / f"part_{number:02}.csv").open(
            "w", encoding="utf-8-sig", newline=""
        ) as output:
            writer = csv.writer(output, delimiter=";")
            writer.writerow(SALES_SCHEMA)
            writer.writerows(rows)


with TemporaryDirectory() as temporary:
    if RU_EXAMPLES.is_dir():
        ru_directory = RU_EXAMPLES
    else:
        ru_directory = Path(temporary)
        write_ru_examples(ru_directory)

    ru_frame = read_csv_directory(ru_directory, blocksize=None)
    ru_result = pd.concat(_partitions(ru_frame, workers=2), ignore_index=True)
    ru_partitions = ru_frame.npartitions

assert len(ru_result) == 4
assert ru_result.loc[0, "product_id"] == "001"
assert ru_result.loc[0, "store_id"] == "005"
assert pd.isna(ru_result.loc[3, "purchase_price"])
print(f"Объединено файлов: 2. Разделов Dask: {ru_partitions}. Строк: {len(ru_result)}.")
display(ru_result)


Объединено файлов: 2. Разделов Dask: 2. Строк: 4.


,product_id,product_name,category,purchase_price,selling_price,quantity_sold,quantity_purchased,losses,date,store_id
0,001,Молоко 1 л,Молочные продукты,54.3200,79.9900,51,60,4,2023-10-02,005
1,002,Хлеб 450 г,Хлебобулочные изделия,32.5000,45.9000,80,90,6,2023-10-02,005
2,001,Молоко 1 л,Молочные продукты,54.3200,79.9900,63,72,5,2023-10-03,013
3,002,Хлеб 450 г,Хлебобулочные изделия,<NA>,45.9000,77,84,6,2023-10-03,013


## Исходные продажи из тестового задания

В папке `data` находится исходный `sales_data.csv`.
Для демонстрации задаём блок 128 КиБ, чтобы увидеть разбиение небольшой выборки.
Просмотр первых пяти строк читает начало данных. Полный контроль количества строк
в следующей ячейке вычисляет все разделы.


In [10]:
sales = read_csv_directory(
    INPUT_DIRECTORY,
    sep=",",
    decimal=".",
    blocksize="128 KiB",
)

assert isinstance(sales, dd.DataFrame)
print(f"Dask DataFrame. Разделов: {sales.npartitions}.")
display(sales.dtypes.rename("Тип Dask").to_frame())
display(sales.head(5))


Dask DataFrame. Разделов: 25.


,Тип Dask
product_id,string
product_name,string
category,category
purchase_price,"decimal128(19, 4)[pyarrow]"
selling_price,"decimal128(19, 4)[pyarrow]"
quantity_sold,int64[pyarrow]
quantity_purchased,int64[pyarrow]
losses,int64[pyarrow]
date,date32[day][pyarrow]
store_id,string


,product_id,product_name,category,purchase_price,selling_price,quantity_sold,quantity_purchased,losses,date,store_id
0,21,Товар_21,Хлебобулочные изделия,381.7700,554.0800,388,932,39,2023-11-22,49
1,12,Товар_12,Овощи,408.5800,536.9900,797,449,32,2023-11-08,100
2,248,Товар_248,Замороженные продукты,314.7500,361.4200,607,106,72,2023-11-07,84
3,129,Товар_129,Кондитерские изделия,314.3400,393.2800,311,743,78,2023-12-14,6
4,463,Товар_463,Овощи,166.4200,200.8900,343,910,78,2023-10-21,8


In [11]:
source_rows = sum(len(partition) for partition in _partitions(sales, workers=2))
assert source_rows == 40_020, "Число строк изменилось относительно исходного файла."
print(f"Прочитаны все разделы. Число строк: {source_rows:,}.".replace(",", " "))


Прочитаны все разделы. Число строк: 40 020.


## Обработка ошибки формата

Проверяем случай, когда CSV объявлен как российский, а цена записана через точку.
Исключение перехватывается и выводится как результат проверки. Аналогично функция
прерывает обработку при повреждённом заголовке, неверной дате и переполнении типа.


In [12]:
with TemporaryDirectory() as temporary:
    bad_directory = Path(temporary)
    broken_row = RU_ROWS[0][0].copy()
    broken_row[3] = "54.32"
    with (bad_directory / "wrong_decimal.csv").open(
        "w", encoding="utf-8-sig", newline=""
    ) as output:
        writer = csv.writer(output, delimiter=";")
        writer.writerow(SALES_SCHEMA)
        writer.writerow(broken_row)

    try:
        invalid = read_csv_directory(bad_directory)
        sum(len(partition) for partition in _partitions(invalid, workers=1))
    except ValueError as error:
        print("Проверка пройдена. Ошибка распознана и перехвачена.")
        print(error)
    else:
        raise AssertionError("Загрузчик пропустил неверный десятичный разделитель.")


Проверка пройдена. Ошибка распознана и перехвачена.
wrong_decimal.csv, purchase_price: неверное число или десятичный разделитель (ожидается ',').


## Полная замена таблицы SQL Server

Каждый раздел загружается пакетами в уникальную служебную таблицу.
После сверки `COUNT_BIG` одна транзакция очищает целевую таблицу через
`TRUNCATE TABLE`, вставляет всю подготовленную выборку и повторно сверяет число
строк. При ошибке замены транзакция откатывается. Если целевой таблицы ещё нет,
она создаётся по заданной схеме.

Существующая SQL-схема должна допускать `TRUNCATE TABLE`. Учётной записи
нужны соответствующие права, а также создание и удаление служебной таблицы.
Блокировка `sp_getapplock` сериализует одновременные запуски этого загрузчика.
При аварийном завершении процесса служебная таблица с префиксом
`__fmcg_stage_` может остаться в базе.


In [13]:
def _table(table: str, db_schema: str, schema: Mapping[str, str]) -> sa.Table:
    if any(
        not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]{0,127}", name)
        for name in (table, db_schema, *schema)
    ):
        raise ValueError("SQL-идентификаторы должны содержать до 128 латинских букв, цифр и _.")
    return sa.Table(
        table,
        sa.MetaData(),
        *(sa.Column(column, SQL_TYPES[kind]) for column, kind in schema.items()),
        schema=db_schema,
        implicit_returning=False,
    )


def _count(connection: Connection, table: sa.Table) -> int:
    query = sa.select(sa.func.count_big(sa.literal_column("*"))).select_from(table)
    return int(connection.execute(query).scalar_one())


def _sql_value(value):
    # Пропуск в category представлен NaN, а в Arrow-колонках используется pd.NA.
    if value is pd.NA or value is pd.NaT or isinstance(value, float) and math.isnan(value):
        return None
    return value


In [14]:
def _refresh_target(
    connection: Connection, stage: sa.Table, target: sa.Table, expected: int
) -> None:
    # Блокировка сериализует обновления загрузчиком, включая создание новой таблицы.
    connection.exec_driver_sql("IF @@TRANCOUNT = 0 BEGIN TRANSACTION")
    lock_status = connection.execute(
        sa.text(
            "DECLARE @status int; "
            "EXEC @status = sys.sp_getapplock @Resource = :resource, "
            "@LockMode = 'Exclusive', @LockOwner = 'Transaction', @LockTimeout = 60000; "
            "SELECT @status;"
        ),
        {"resource": f"load_csv:{target.schema}.{target.name}".lower()},
    ).scalar_one()
    if lock_status < 0:
        raise RuntimeError(f"Не получена блокировка обновления (код {lock_status}).")

    if sa.inspect(connection).has_table(target.name, schema=target.schema):
        qualified = connection.dialect.identifier_preparer.format_table(target)
        connection.exec_driver_sql(f"TRUNCATE TABLE {qualified}")
    else:
        target.create(connection)
    columns = list(target.columns.keys())
    connection.execute(target.insert().from_select(columns, sa.select(*stage.columns)))
    actual = _count(connection, target)
    if actual != expected:
        raise RuntimeError(f"Контроль целевой таблицы: ожидалось {expected} строк, записано {actual}.")


In [15]:
def replace_sql_table(
    frame: dd.DataFrame,
    engine: Engine,
    table: str,
    *,
    db_schema: str = "dbo",
    schema: Mapping[str, str] = SALES_SCHEMA,
    batch_rows: int = 10_000,
    workers: int = 2,
) -> int:
    """Сохранить разделы в staging, затем атомарно заменить целевую таблицу."""
    if engine.dialect.name != "mssql":
        raise ValueError("Ожидается подключение к SQL Server.")
    if batch_rows < 1 or workers < 1:
        raise ValueError("Размер партии и число рабочих потоков должны быть положительными.")
    if list(frame.columns) != list(schema):
        raise ValueError("Колонки Dask DataFrame не совпадают со схемой загрузки.")

    target = _table(table, db_schema, schema)
    stage = _table(f"__fmcg_stage_{uuid4().hex}", db_schema, schema)
    with engine.begin() as connection:
        stage.create(connection)

    written = 0
    try:
        insert = stage.insert()
        for partition in _partitions(frame, workers):
            rows = partition.itertuples(index=False, name=None)
            while batch := list(itertools.islice(rows, batch_rows)):
                parameters = [
                    {column: _sql_value(value) for column, value in zip(schema, row)}
                    for row in batch
                ]
                with engine.begin() as connection:
                    connection.execute(insert, parameters)
                written += len(parameters)

        with engine.begin() as connection:
            actual = _count(connection, stage)
            if actual != written:
                raise RuntimeError(f"Контроль staging: прочитано {written} строк, записано {actual}.")
            _refresh_target(connection, stage, target, written)
    finally:
        try:
            with engine.begin() as connection:
                stage.drop(connection)
        except SQLAlchemyError:
            LOG.warning("Не удалось удалить служебную таблицу %s.%s.", db_schema, stage.name)
    return written


Строку подключения задайте в переменной окружения `FMCG_SQL_ODBC` перед запуском
Jupyter. Например, для Windows-аутентификации она содержит
`Driver={ODBC Driver 18 for SQL Server};Server=server;Database=SalesSandbox;Trusted_Connection=yes;Encrypt=yes;`.

Чтобы выполнить запись, укажите свою базу и целевую таблицу, затем установите
`RUN_SQL = True` в ячейке параметров. Полная замена применяется к
`SQL_SCHEMA.SQL_TABLE`. Если входные CSV содержат только заголовки,
результатом замены будет пустая таблица.


In [16]:
if RUN_SQL:
    connection_string = os.environ.get("FMCG_SQL_ODBC")
    if not connection_string:
        raise ValueError("Задайте строку подключения в переменной FMCG_SQL_ODBC.")

    engine = sa.create_engine(
        URL.create("mssql+pyodbc", query={"odbc_connect": connection_string}),
        fast_executemany=True,
        hide_parameters=True,
        pool_pre_ping=True,
    )
    try:
        uploaded_rows = replace_sql_table(
            sales,
            engine,
            SQL_TABLE,
            db_schema=SQL_SCHEMA,
            batch_rows=10_000,
            workers=2,
        )
        print(f"Обновлена таблица {SQL_SCHEMA}.{SQL_TABLE}. Строк: {uploaded_rows}.")
    except SQLAlchemyError as error:
        original_args = getattr(getattr(error, "orig", None), "args", ())
        sql_state = str(original_args[0]) if original_args else ""
        detail = f", SQLSTATE {sql_state}" if re.fullmatch(r"[0-9A-Z]{5}", sql_state) else ""
        raise RuntimeError(
            f"Ошибка SQL Server ({type(error).__name__}{detail}). "
            "Проверьте доступность сервера, права и схему таблицы."
        ) from None
    finally:
        engine.dispose()
else:
    print("RUN_SQL = False. Чтение и проверка CSV выполнены. Подключение к SQL Server отключено.")


RUN_SQL = False. Чтение и проверка CSV выполнены. Подключение к SQL Server отключено.


## Результат проверки

Объединены два CSV с десятичной запятой, сохранены ведущие нули и пропуск цены.
Исходный файл тестового прочитан полностью, получено **40 020 строк**.
Неверный десятичный разделитель распознан управляемой проверкой.

Код записи и полной замены приведён выше. Реальный SQL Server при подготовке
этого notebook не подключался.

Технические детали поведения библиотек и SQL Server приведены в
[Dask read_csv](https://docs.dask.org/en/stable/generated/dask.dataframe.read_csv.html),
[SQLAlchemy fast_executemany](https://docs.sqlalchemy.org/en/20/dialects/mssql.html#fast-executemany-mode)
и [TRUNCATE TABLE](https://learn.microsoft.com/en-us/sql/t-sql/statements/truncate-table-transact-sql).
